In [ ]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec

In [ ]:
import os
import random
import torch
import torchaudio
import numpy as np


from torch.utils.data import Dataset
from transformers import (
Wav2Vec2Processor,
Wav2Vec2ForSequenceClassification,
TrainingArguments,
DataCollatorWithPadding,
Trainer
)


from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
# -------------------------
# Configuración general
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/dataset"
SAMPLE_RATE = 16000
NUM_LABELS = 3
SEED = 42


random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# -------------------------
# Mapeo de emociones (3 clases)
# -------------------------
emotion2id = {
    "alegria": 0,      # Positivo
    "neutro": 1,      # Neutral
    "tristeza": 2,     # Negativo
    "miedo": 2,
    "sorpresa": 2,
    "ira": 2,
    "asco": 2
}

id2emotion = {
    0: "positivo",
    1: "neutral",
    2: "negativo"
}


In [ ]:
# -------------------------
# Dataset
# -------------------------
class EmotionDataset(Dataset):
    def __init__(self, files, data_dir, processor):
        self.files = files
        self.data_dir = data_dir
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.data_dir, filename)

        # Etiqueta desde el nombre
        emotion = filename.replace(".WAV","").replace(".wav","").split("_")[1].lower()

        label = emotion2id[emotion]

        # Cargar audio
        speech, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            speech = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(speech)

        speech = speech.mean(dim=0)  # mono

        inputs = self.processor(
            speech.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

In [ ]:
# -------------------------
# Modelo y procesador
# -------------------------
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=processor,
    padding=True,
    return_tensors="pt"
)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(DEVICE)

# Congelar extractor (clave para corpus pequeño)
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-spanish
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
projector.bias    | MISSING    | 
classifier.bias   | MISSING    | 
projector.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# -------------------------
# División del corpus
# -------------------------
all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(".wav")]
random.shuffle(all_files)

print("Archivos detectados:", len(all_files))
print(all_files[:5])


train_size = int(0.8 * len(all_files))
train_files = all_files[:train_size]
test_files = all_files[train_size:]

print(f"Train: {len(train_files)} | Test: {len(test_files)}")

train_dataset = EmotionDataset(train_files, DATASET_PATH, processor)
test_dataset = EmotionDataset(test_files, DATASET_PATH, processor)


Archivos detectados: 2004
['07_Miedo_02.WAV', '46_Neutro_11.WAV', '34_Sorpresa_01.WAV', '22_Tristeza_09.WAV', '37_Alegria_10.WAV']
Train: 1603 | Test: 401


In [ ]:
# -------------------------
# Métricas
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }

In [ ]:
# -------------------------
# Argumentos de entrenamiento
# -------------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    fp16=False
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# -------------------------
# Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
# -------------------------
# Entrenamiento y evaluación
# -------------------------
trainer.train()

results = trainer.evaluate()
print("\nResultados finales:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,1.015985,1.032069,0.753117,0.645946,0.753117,0.693453
2,0.870860,0.859582,0.807980,0.693318,0.807980,0.744157
3,0.006413,0.947826,0.825436,0.849528,0.825436,0.773320
4,3.197405,0.685346,0.852868,0.843733,0.852868,0.846205
5,1.152023,0.869494,0.860349,0.853222,0.860349,0.854152
6,1.219123,0.731528,0.890274,0.887229,0.890274,0.884885
7,0.000763,0.727224,0.890274,0.890910,0.890274,0.890441
8,1.430761,0.666197,0.907731,0.906139,0.907731,0.906519


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Resultados finales:
eval_loss: 0.6662
eval_accuracy: 0.9077
eval_precision: 0.9061
eval_recall: 0.9077
eval_f1-score: 0.9065
eval_runtime: 23.2172
eval_samples_per_second: 17.2720
eval_steps_per_second: 17.2720
epoch: 8.0000
